# 05 - XGBoost Forecasting Model

## Objective

Evaluate XGBoost regression using lagged revenue, growth, quantity, and price features. Features are built only from previous months to avoid using information from the forecast period.

In [ ]:
from pathlib import Path
import pandas as pd

# Caminho of the project
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "interm"

# Load the data
csv_file = DATA_PATH / "data_3.csv"
banco = pd.read_csv(csv_file)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from xgboost import XGBRegressor

In [ ]:
# Parâmetros of estilo of the Gráfico based in the manual of the MBA
plt.rcParams.update({

    # font
    "font.family": "Arial",
    "font.size": 11,

    # background
    "figure.facecolor": "white",
    "axes.facecolor": "white",

    # without grid
    "axes.grid": False,

    # without title
    "axes.titlesize": 0,

    # axes
    "axes.edgecolor": "black",
    "axes.linewidth": 1.5,

    # color of the textos
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",

    # size of the titles of thexes
    "axes.labelsize": 11

})

In [ ]:
def plot_with_style(
    data,
    kind='line',                 # 'line' or 'bar'
    x_label='',
    y_label='',
    title='',
    xticks=None,                 # custom ticks
    xtlabels=None,               # labels personalizados
    rotate_xticks=0,
    figsize=(12, 6),
    marker='o',
    linewidth=2,
    colors=None,
    linestyles=None,
    xlim=None,                   # e.g., ('2017-01-01', '2026-01-01')
    year_ticks=True,             # ticks annual automáticos
    vline=None,                  # e.g., test.index[0]
    vline_label=None
):

    # ---------------------------------------------------------
    # Validation
    # ---------------------------------------------------------

    if not isinstance(data, (pd.Series, pd.DataFrame)):
        raise ValueError(
            'date must be pd.Series or pd.DataFrame'
        )

    if kind not in ['line', 'bar']:
        raise ValueError(
            "kind must be 'line' or 'bar'"
        )

    # ---------------------------------------------------------
    # Create the figure
    # ---------------------------------------------------------

    fig, ax = plt.subplots(figsize=figsize)

    # ---------------------------------------------------------
    # Gráfico of Series
    # ---------------------------------------------------------

    if isinstance(data, pd.Series):

        label = data.name if data.name is not None else 'Series'

        if kind == 'line':

            ax.plot(
                data.index,
                data.values,
                marker=marker,
                linewidth=linewidth,
                label=label
            )

        else:

            ax.bar(
                data.index,
                data.values,
                edgecolor='black',
                linewidth=0.8,
                label=label
            )

    # ---------------------------------------------------------
    # Gráfico of DataFrame
    # ---------------------------------------------------------

    elif isinstance(data, pd.DataFrame):

        if kind == 'line':

            for i, column in enumerate(data.columns):

                kwargs = {
                    'marker': marker,
                    'linewidth': linewidth,
                    'label': column
                }

                if colors is not None:
                    kwargs['color'] = colors[i]

                if linestyles is not None:
                    kwargs['linestyle'] = linestyles[i]

                ax.plot(
                    data.index,
                    data[column],
                    **kwargs
                )

        else:

            data.plot(
                kind='bar',
                ax=ax
            )

    # ---------------------------------------------------------
    # labels
    # ---------------------------------------------------------

    ax.set_xlabel(
        x_label,
        fontsize=11,
        fontname='Arial'
    )

    ax.set_ylabel(
        y_label,
        fontsize=11,
        fontname='Arial'
    )

    ax.set_title(
        title,
        fontsize=13,
        fontname='Arial'
    )

    # ---------------------------------------------------------
    # Estilo of thexes
    # ---------------------------------------------------------

    ax.grid(False)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.spines['left'].set_visible(True)
    ax.spines['bottom'].set_visible(True)

    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)

    ax.spines['left'].set_color('black')
    ax.spines['bottom'].set_color('black')

    # ---------------------------------------------------------
    # Limites of thexis X
    # ---------------------------------------------------------

    if xlim is not None:

        if isinstance(data.index, pd.DatetimeIndex):

            xlim = (
                pd.to_datetime(xlim[0]),
                pd.to_datetime(xlim[1])
            )

        ax.set_xlim(xlim)

    # ---------------------------------------------------------
    # Ticks temporais annual
    # ---------------------------------------------------------

    if (
        year_ticks
        and isinstance(data.index, pd.DatetimeIndex)
        and xticks is None
    ):

        ax.xaxis.set_major_locator(
            mdates.YearLocator()
        )

        ax.xaxis.set_major_formatter(
            mdates.DateFormatter('%Y')
        )

    # ---------------------------------------------------------
    # Ticks personalizados
    # ---------------------------------------------------------

    if xticks is not None:

        ax.set_xticks(xticks)

    if xtlabels is not None:

        ax.set_xticklabels(
            xtlabels,
            fontname='Arial'
        )

    # ---------------------------------------------------------
    # Tick rotation
    # ---------------------------------------------------------

    if rotate_xticks:

        plt.setp(
            ax.get_xticklabels(),
            rotation=rotate_xticks,
            ha='right'
        )

    # ---------------------------------------------------------
    # Vertical reference line (ex.: Start of test period)
    # ---------------------------------------------------------

    if vline is not None:

        ax.axvline(
            vline,
            linestyle='--',
            linewidth=1.5,
            label=vline_label,
            c='r'
        )

    # ---------------------------------------------------------
    # legend
    # ---------------------------------------------------------

    handles, labels = ax.get_legend_handles_labels()

    if len(labels) > 1:
        ax.legend(
            title=None,
            frameon=False
        )

    # ---------------------------------------------------------
    # Layout
    # ---------------------------------------------------------

    plt.tight_layout()

    plt.show()

## XGBoost Baseline

To forecast revenue at month $t$, the model must use information available before month $t$. Current-month quantity and price are therefore excluded from the baseline.

In [ ]:
# Copiar the dataframe original
df = banco.copy()

# Convertr sale date for datetime
df['data_venda'] = pd.to_datetime(df['data_venda'])

# create year and month
df['ano'] = df['data_venda'].dt.year
df['mes'] = df['data_venda'].dt.month

# revenue total by month
df_mensal = df.groupby(['ano', 'mes']).agg(
    valor_total=('valor_total', 'sum'),
    quantidade_total=('quantidade', 'sum')
).reset_index()

df_mensal.head()

In [ ]:
split = int(len(df_mensal) * 0.8)

## 01 - Naive Revenue Lag 1

In [ ]:
# create column of revenue total of the month previous
df_naive = df_mensal.copy()
df_naive['faturamento_lag1'] = df_naive['valor_total'].shift(1)
df_naive.dropna(inplace=True)  # Remove the first row containing NaN

df_naive.head()

In [ ]:
X = df_naive.drop(columns=['valor_total', 'quantidade_total'])
y = df_naive['valor_total']

X.head()

In [ ]:
X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

In [ ]:
xgb_naive = XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42,
    objective='reg:squarederror'  # Added to avoid a deprecation warning
)

xgb_naive.fit(X_train, y_train)

y_pred_xgb_naive = xgb_naive.predict(X_test)

In [ ]:
mae_xgb_naive = mean_absolute_error(y_test, y_pred_xgb_naive)

rmse_xgb_naive = np.sqrt(
    mean_squared_error(y_test, y_pred_xgb_naive)
)

print(f'MAE - XGBoost Naive (lag1): {mae_xgb_naive:.2f}')
print(f'RMSE - XGBoost Naive (lag1): {rmse_xgb_naive:.2f}')

## 02 - Recent Revenue Dynamics: First Difference

`diff1` measures the short-term change between consecutive months: $\Delta y_t = y_t - y_{t-1}$

In [ ]:
df_02 = df_naive.copy()

df_02['diff1'] = df_02['valor_total'].diff(1)
df_02.dropna(inplace=True)  # Remove the first row containing NaN

df_02.head()

In [ ]:
X = df_02.drop(columns=['quantidade_total', 'valor_total'])
y = df_02['valor_total']

X.head()

In [ ]:
X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

In [ ]:
xgb_02 = XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42,
    objective='reg:squarederror'  # Added to avoid a deprecation warning
)

xgb_02.fit(X_train, y_train)

y_pred_xgb_02 = xgb_02.predict(X_test)

In [ ]:
mae_xgb_02 = mean_absolute_error(y_test, y_pred_xgb_02)

rmse_xgb_02 = np.sqrt(
    mean_squared_error(y_test, y_pred_xgb_02)
)

print(f'MAE - XGBoost 02: {mae_xgb_02:.2f}')
print(f'RMSE - XGBoost 02: {rmse_xgb_02:.2f}')

In [ ]:
# Create dates for the time axis
datas = pd.to_datetime(
    df_02['ano'].astype(str) + '-' +
    df_02['mes'].astype(str) + '-01'
)

# DataFrame for the plot
grafico_xgb_02 = pd.DataFrame(
    index=datas
)

# Initialize the series
grafico_xgb_02['training'] = np.nan
grafico_xgb_02['Actual'] = np.nan
grafico_xgb_02['Forecast - XGBoost'] = np.nan

# Fill the training series
grafico_xgb_02.iloc[:split, grafico_xgb_02.columns.get_loc('training')] = y_train.values

# Fill actual test values
grafico_xgb_02.iloc[split:, grafico_xgb_02.columns.get_loc('Actual')] = y_test.values

# Fill forecast values
grafico_xgb_02.iloc[split:, grafico_xgb_02.columns.get_loc('Forecast - XGBoost')] = y_pred_xgb_02

In [ ]:
plot_with_style(
    grafico_xgb_02,
    x_label='year',
    y_label='revenue (R$)',
    figsize=(12, 6),
    marker=None,
    linewidth=2,
    xlim=('2017-01-01', '2026-01-01'),
    year_ticks=True,
    vline=grafico_xgb_02.index[split],
    vline_label='Start of test period'
)

## 03 - Growth Dynamics

$g_t = \frac{y_{t-1}}{y_{t-2}} - 1$

In [ ]:
df_03 = df_naive.copy()

def safe_growth(numerator, denominator):
    return np.where(
        denominator != 0,
        numerator / denominator - 1,
        np.nan
    )

prev_1 = df_03['valor_total'].shift(1)
prev_2 = df_03['valor_total'].shift(2)

df_03['growth_1'] = safe_growth(
    prev_1,
    prev_2
)

df_03.dropna(inplace=True)

df_03.head()

In [ ]:
X = df_03.drop(columns=['quantidade_total', 'valor_total', 'faturamento_lag1'])
y = df_03['valor_total']

X.head()

In [ ]:
X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

In [ ]:
xgb_03 = XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42,
    objective='reg:squarederror'  # Added to avoid a deprecation warning
)

xgb_03.fit(X_train, y_train)

y_pred_xgb_03 = xgb_03.predict(X_test)

In [ ]:
mae_xgb_03 = mean_absolute_error(y_test, y_pred_xgb_03)

rmse_xgb_03 = np.sqrt(
    mean_squared_error(y_test, y_pred_xgb_03)
)

print(f'MAE - XGBoost 03: {mae_xgb_03:.2f}')
print(f'RMSE - XGBoost 03: {rmse_xgb_03:.2f}')

The growth feature performs worse than the first-difference feature. This suggests that the model benefits more from recent absolute changes than from a relative growth rate.

## 04 - XGBoost with Quarterly, Semiannual, and Annual Lags

In [ ]:
df_04 = df_naive.copy()

for lag in [1, 2, 3, 6, 12]:
    df_04[f'lag_{lag}'] = (
        df_04['valor_total'].shift(lag)
    )
df_04.dropna(inplace=True)

df_04.head()

In [ ]:
X = df_04.drop(columns=['quantidade_total', 'valor_total', 'faturamento_lag1', 'lag_2', 'lag_3'])
y = df_04['valor_total']

X.head()

In [ ]:
X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

In [ ]:
xgb_04 = XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42,
    objective='reg:squarederror'  # Added to avoid a deprecation warning
)

xgb_04.fit(X_train, y_train)

y_pred_xgb_04 = xgb_04.predict(X_test)

In [ ]:
mae_xgb_04 = mean_absolute_error(y_test, y_pred_xgb_04)

rmse_xgb_04 = np.sqrt(
    mean_squared_error(y_test, y_pred_xgb_04)
)

print(f'MAE - XGBoost 04: {mae_xgb_04:.2f}')
print(f'RMSE - XGBoost 04: {rmse_xgb_04:.2f}')

The lag experiments do not show a consistent seasonal improvement. The best results use a small set of recent lags, suggesting that long seasonal lags add limited predictive value for this dataset.

## 05 - Combining Lag 1, Growth 1, and Difference 1

This model combines recent level, relative growth, and absolute change features.

In [ ]:
df_05 = df_03.copy()

df_05['diff1'] = df_05['valor_total'].diff(1)
df_05.dropna(inplace=True)  # Remove the first row containing NaN

df_05.head()

In [ ]:
X = df_05.drop(columns=['quantidade_total', 'valor_total'])
y = df_05['valor_total']

X.head()

In [ ]:
X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

In [ ]:
xgb_05 = XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42,
    objective='reg:squarederror'  # Added to avoid a deprecation warning
)

xgb_05.fit(X_train, y_train)

y_pred_xgb_05 = xgb_05.predict(X_test)

In [ ]:
mae_xgb_05 = mean_absolute_error(y_test, y_pred_xgb_05)

rmse_xgb_05 = np.sqrt(
    mean_squared_error(y_test, y_pred_xgb_05)
)

print(f'MAE - XGBoost 05: {mae_xgb_05:.2f}')
print(f'RMSE - XGBoost 05: {rmse_xgb_05:.2f}')

The model performs best with features that capture rapid changes over a short horizon. Adding growth does not improve the model enough to offset the additional complexity.

## 06 - Understanding the Role of Quantity

In [ ]:
df_06 = df_naive.copy()

df_06['quantidade_lag1'] = df_06['quantidade_total'].shift(1)
df_06.dropna(inplace=True)  # Remove the first row containing NaN

df_06.head()

In [ ]:
X = df_06.drop(columns=['quantidade_total', 'valor_total'])
y = df_06['valor_total']

X.head()

In [ ]:
X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

In [ ]:
xgb_06 = XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42,
    objective='reg:squarederror'  # Added to avoid a deprecation warning
)

xgb_06.fit(X_train, y_train)

y_pred_xgb_06 = xgb_06.predict(X_test)

In [ ]:
mae_xgb_06 = mean_absolute_error(y_test, y_pred_xgb_06)

rmse_xgb_06 = np.sqrt(
    mean_squared_error(y_test, y_pred_xgb_06)
)

print(f'MAE - XGBoost Revenue and Quantity Lag 1: {mae_xgb_06:.2f}')
print(f'RMSE - XGBoost Revenue and Quantity Lag 1: {rmse_xgb_06:.2f}')

## 07 - Understanding the Role of Price and Quantity

In [ ]:
df['valor_ponderado'] = (
    df['valor_unitario'] * df['quantidade']
)

df_07 = (
    df.groupby(['ano', 'mes'])
      .agg(
        valor_total=('valor_total', 'sum'),
        quantidade_total=('quantidade', 'sum'),
        valor_ponderado=('valor_ponderado', 'sum')
      )
      .reset_index()
)

df_07['preco_medio'] = (
    df_07['valor_ponderado'] / df_07['quantidade_total']
)

df_07['quantidade_lag1'] = df_07['quantidade_total'].shift(1)
df_07['preco_medio_lag1'] = df_07['preco_medio'].shift(1)
df_07['faturamento_lag1'] = df_07['valor_total'].shift(1)

df_07.dropna(inplace=True)  # Remove the first row containing NaN
df_07.head()

In [ ]:
X = df_07.drop(columns=['valor_total', 'valor_ponderado', 'preco_medio', 'quantidade_total'])
y = df_07['valor_total']
X.head()

In [ ]:
X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

In [ ]:
xgb_07 = XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42,
    objective='reg:squarederror'  # Added to avoid a deprecation warning
)

xgb_07.fit(X_train, y_train)

y_pred_xgb_07 = xgb_07.predict(X_test)

In [ ]:
mae_xgb_07 = mean_absolute_error(y_test, y_pred_xgb_07)

rmse_xgb_07 = np.sqrt(
    mean_squared_error(y_test, y_pred_xgb_07)
)

print(f'MAE - XGBoost 07 (revenue/quantity/average price lag 1): {mae_xgb_07:.2f}')
print(f'RMSE - XGBoost 07 (revenue/quantity/average price lag 1): {rmse_xgb_07:.2f}')